# Fenic Playground (No API Key)

This notebook demonstrates **fenic-style workflows** using a local playground that requires **no API key**.

- Uses a `MockLLM` with canned/demo responses
- Shows async UDF-style processing, Pydantic validation, and semantic joins
- Lets you experiment with the core ideas of fenic without external dependencies

To run: just execute all cells below.

## Installation

Install dependencies (in your environment):

In [ ]:
!pip install pydantic

## Playground Overview

The playground provides:
- A `MockLLM` that returns canned or auto-generated responses
- Async batch processing for extraction
- Pydantic schema validation
- A minimal DataFrame-like wrapper (`MiniDF`)
- A mock semantic join based on similarity

No API keys or cloud access required.

In [ ]:
# If running in Colab or a different directory, adjust the path as needed
import sys
sys.path.append("/fenic/examples/playground")

## Playground Code

Let's load the playground code and canned responses.

In [ ]:
# Load playground.py and canned_responses.json
from playground import MockLLM, MiniDF, Extracted, DEMO_DOCS, METADATA
import json
import os

canned_path = "/fenic/examples/playground/canned_responses.json"
if os.path.exists(canned_path):
    with open(canned_path, "r", encoding="utf-8") as f:
        canned = json.load(f)
else:
    canned = {}

## Demo Dataset

We'll use a small set of demo documents and metadata for extraction and joining.

In [ ]:
print("Demo documents:")
for row in DEMO_DOCS:
    print(row)
print("\nMetadata:")
for row in METADATA:
    print(row)

## Semantic Extraction (Async, Pydantic Validated)

We'll extract structured information from each document using the mock LLM and validate with Pydantic.

In [ ]:
import asyncio

llm = MockLLM(canned=canned)
df = MiniDF(DEMO_DOCS)

async def run_extract():
    results = await df.semantic_extract(column="text", schema=Extracted, llm=llm, batch_size=2)
    for res in results:
        if res.get("ok"):
            v = res["value"]
            print(f"Row {res['row']} -> title={v.title!r}, score={v.score}")
        else:
            print("Error:", res)

await run_extract()

## Semantic Join (Similarity-Based)

Now, let's join the demo documents to the metadata using mock similarity scoring.

In [ ]:
meta = MiniDF(METADATA)

async def run_join():
    joined = await df.semantic_join(meta, left_column="text", right_column="content", llm=llm)
    for j in joined:
        left_id = j["left"]["id"]
        right_title = j["right"]["title"] if j["right"] else "<no match>"
        print(f"Left id={left_id} matched right title={right_title} (score={j['score']})")

await run_join()

## Playground Metrics

You can inspect the number of mock LLM calls and average latency.

In [ ]:
metrics = llm.metrics()
print(f"LLM calls: {metrics['calls']}, avg call latency: {metrics['avg_latency_s']:.4f}s, total LLM latency: {metrics['total_latency_s']:.3f}s")

## Summary

- This playground lets you try fenic-style workflows locally, with no API key or cloud access.
- You can experiment with async extraction, Pydantic validation, and semantic joins.
- To extend, modify the demo dataset or canned responses, or try your own text inputs.

For more advanced features, see the main [fenic documentation](https://docs.fenic.ai) or try the full API with a real provider.